In [1]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
import plotly.io as pio
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
from sklearn.metrics import precision_recall_curve
import sys
print(sys.executable)

c:\Users\Kasia\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [4]:
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

In [5]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [10]:
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint
from scipy.optimize import differential_evolution

# For Bayesian Optimization
# Install if needed: pip install scikit-optimize
from skopt import gp_minimize
from skopt.space import Integer
from skopt.utils import use_named_args

In [6]:
# ---------------------------------------------------------------------
# loading data
# ---------------------------------------------------------------------
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    return p

REPO_ROOT = find_repo_root()

In [7]:
# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
REPO_ROOT    = Path(r"F:\Apps\Credit-Risk-Score-ML")
DATA_PATH    = REPO_ROOT / "data" / "raw" / "default of credit card clients.xls"
RANDOM_STATE = 42

print("Repo root:", REPO_ROOT)
print("Data path:", DATA_PATH)
print("Exists:   ", DATA_PATH.exists())

Repo root: F:\Apps\Credit-Risk-Score-ML
Data path: F:\Apps\Credit-Risk-Score-ML\data\raw\default of credit card clients.xls
Exists:    True


In [8]:
# ---------------------------------------------------------------------
# LOAD RAW DATA 
# ---------------------------------------------------------------------
df_raw = pd.read_excel(DATA_PATH, sheet_name=0, engine="xlrd", header=1)
print(df_raw.columns.tolist())
df_raw.head()

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [9]:
# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------
def print_df(df: pd.DataFrame, name: str, n: int = 10):
    print("\n" + "-" * 95)
    print(f"[DATAFRAME] {name} | shape = {df.shape[0]:,} rows × {df.shape[1]:,} cols")
    print("-" * 95)
    print(df.head(n).to_string())

def section(title: str):
    print("\n" + "=" * 95)
    print(title)
    print("=" * 95)

def pct(x):
    return 100 * x

In [14]:
print(df_raw.columns.tolist())

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']
